# Shor's Algorithm — Factoring N = 15 and N = 21

A runnable walkthrough of the implementation in [`shor_factoring.py`](shor_factoring.py).

Shor's algorithm is **one quantum subroutine (order-finding) wrapped in a classical loop**.
We first factor **N = 15** with the hand-built swap-network gates, then factor
**N = 21** with the general permutation-unitary gates, and finish with a base comparison
that shows why a small-order base is the efficient choice.

Everything runs on the local `AerSimulator`; the harness is backend-agnostic.

In [ ]:
from fractions import Fraction
from math import gcd

from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

from shor_factoring import (
    c_amod15,
    c_amod,
    inverse_qft,
    build_order_finding_circuit,
    run_circuit_and_get_counts,
    order_from_counts,
    find_order,
    shor,
    analyze_base,
)

backend = AerSimulator()
backend

## Part 1 — N = 15

## 1.1 The order-finding circuit

8 counting qubits + 4 work qubits = 12 qubits. Each `count[k]` controls
`a^(2^k) mod 15` on the work register, then a hand-built inverse QFT maps the
accumulated phase back to a measurable integer.

In [ ]:
a = 7
qc = build_order_finding_circuit(a, N=15, n_count=8)
print(qc.num_qubits, 'qubits,', qc.size(), 'gates (pre-transpile)')
qc.draw('mpl', fold=-1)

## 1.2 Phase peaks

For an order-`r` base the measured values cluster at multiples of `256 / r`.
For `a = 7` (order 4) that means clean spikes at **0, 64, 128, 192**.

In [ ]:
counts = run_circuit_and_get_counts(qc, backend, shots=2048)
decimal = {str(int(b, 2)): cnt for b, cnt in counts.items()}
plot_histogram(decimal, title=f'N=15 counting-register readout for a={a}')

## 1.3 Decode the order, then factor

Continued fractions turn each measured phase `s/256` into the order `r`. Once `r`
is even and `a^(r/2) != -1 mod 15`, the factors are `gcd(a^(r/2) +/- 1, 15)`.

In [ ]:
r = order_from_counts(counts, a, N=15, n_count=8)
print(f'recovered order of {a} mod 15: r = {r}')
x = pow(a, r // 2, 15)
print('factors:', gcd(x - 1, 15), 'x', gcd(x + 1, 15))

import random
random.seed(2)
print('shor(backend, N=15) ->', shor(backend, N=15))

## Part 2 — N = 21

For N = 21 the residues need 5 work qubits and the modular multiplication no
longer collapses to a tidy swap network, so we build the controlled map
`|y> -> |a^(2^k) * y mod 21>` directly as a permutation unitary (`c_amod`).
The classical wrapper and inverse QFT are unchanged.

We use `a = 8`, whose order mod 21 is just **2**, so 3 counting qubits suffice and
the peaks fall at **0 and 4** (multiples of `8 / 2`).

In [ ]:
res21 = analyze_base(8, backend, N=21, n_count=3)
print('a=8 mod 21  ->  order', res21['order'], ' factors', res21['factors'])
print('total qubits', res21['num_qubits'], ' transpiled depth', res21['depth'])
decimal21 = {str(int(b, 2)): cnt for b, cnt in res21['counts'].items()}
plot_histogram(decimal21, title='N=21 counting-register readout for a=8 (order 2)')

In [ ]:
# Full algorithm end-to-end on N = 21
random.seed(2)
print('shor(backend, N=21) ->', shor(backend, N=21, n_count=8))

## Part 3 — Base comparison for N = 21

Two coprime bases that both factor `21 = 3 x 7`. `a = 8` has order 2 (minimal);
`a = 2` has order 6 (the full six-peak period spectrum). Smaller order means
fewer counting qubits and a much shallower circuit.

In [ ]:
import matplotlib.pyplot as plt

r8 = analyze_base(8, backend, N=21, n_count=3)
r2 = analyze_base(2, backend, N=21, n_count=5)

print('Comparison of two coprime bases for factoring N = 21')
print('-' * 58)
print(f"{'metric':<20}{'a = 8':<16}{'a = 2'}")
print('-' * 58)
for label, key in [('order r','order'), ('factors found','factors'),
                   ('counting qubits','n_count'), ('total qubits','num_qubits'),
                   ('circuit depth','depth'), ('distinct outcomes','distinct_outcomes')]:
    print(f'{label:<20}{str(r8[key]):<16}{r2[key]}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 5))
plot_histogram({str(int(b,2)): c for b,c in r8['counts'].items()}, ax=ax1,
               title=f"a=8, order r={r8['order']} -> factors {r8['factors']}")
plot_histogram({str(int(b,2)): c for b,c in r2['counts'].items()}, ax=ax2,
               title=f"a=2, order r={r2['order']} -> factors {r2['factors']}")
plt.tight_layout()
plt.show()

## Part 4 — Running on real IBM hardware

The harness is backend-agnostic — swap the backend and nothing else changes:

```python
from qiskit_ibm_runtime import QiskitRuntimeService
service = QiskitRuntimeService()           # after save_account(...) once
hw = service.least_busy(operational=True, simulator=False)
print(shor(hw, N=15))
```

Expect the phase peaks to smear on current NISQ devices: validate on the
simulator, then run on hardware and report the degradation.